# Модуль обучения случайного леса

In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv("data\\train.csv", index_col='id')

Применяем те же преобразования, что и в show_data.ipynb и исключим признаки, которые имеют много категориальных значений:

In [3]:
from sklearn.preprocessing import FunctionTransformer

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.fillna({'Profession': 'unemployed'}, inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

Применяем OneHot энкодинг для категориальных признаков

In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

encoder = ColumnTransformer([
    ('onehot', OneHotEncoder(), make_column_selector(dtype_include=object))
])

Оценивать качество будем при помощи кросс-валидации

In [8]:
from sklearn.model_selection import cross_val_score

def cv(model, X, y):
    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1'
    )

    return score.mean()        

In [9]:
X = df_train.iloc[:, :-1]
y = df_train.iloc[:, -1]

Строим пайплайн и оцениваем качество

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('prep', prep),
    ('encode', encoder),
    ('model', RandomForestClassifier()),
])

cv(pipe, X, y)

np.float64(0.500375813631998)

In [12]:
pipe.fit(X, y)
pipe.named_steps['model'].feature_importances_

array([0.02838152, 0.02846423, 0.07786214, 0.06893203, 0.00215388,
       0.00629508, 0.00503475, 0.00196315, 0.001748  , 0.0025485 ,
       0.00303827, 0.00408334, 0.00285562, 0.00446622, 0.00015258,
       0.00323998, 0.00232936, 0.00302626, 0.00252255, 0.0015369 ,
       0.00287004, 0.00490821, 0.00420429, 0.00019726, 0.00549745,
       0.00387262, 0.00309666, 0.00238072, 0.00426326, 0.00247331,
       0.00112635, 0.0024737 , 0.00239213, 0.00216535, 0.00268564,
       0.00073574, 0.01222991, 0.00134741, 0.00143307, 0.0814047 ,
       0.02708717, 0.02347236, 0.03201909, 0.02580053, 0.02323491,
       0.02503015, 0.02977761, 0.01124682, 0.01365407, 0.01065073,
       0.00839691, 0.00902987, 0.00881671, 0.00883284, 0.0106058 ,
       0.00685967, 0.00879558, 0.00851418, 0.03508256, 0.01137911,
       0.00580883, 0.00613554, 0.00658028, 0.00754242, 0.0081322 ,
       0.01066928, 0.00524617, 0.00628074, 0.0142764 , 0.00919464,
       0.00528101, 0.00553114, 0.00663397, 0.00967682, 0.05114

попробуем воспользоваться GridSearchCV для перебора гиперпараметров

In [14]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('prep', prep),
    ('encode', encoder),
    ('model', RandomForestClassifier()),
])

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [5, 10, None],
    "model__max_features": ["sqrt", "log2"],
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring='f1',
    cv=5,
)

grid.fit(X, y)

grid.best_score_

np.float64(0.547301841756916)

Качество получилось все равно значительно ниже, чем у моделей линейной регрессии и метода опорных векторов. 